# bci-eeg-to-text

CTC-based EEG-to-text decoding. This notebook runs the parts that need **no dataset**, so it works on a hosted CPU image.

The EEG corpora are not bundled: they are 419 GB across two OpenNeuro datasets. Dataset-backed tests skip themselves automatically.


In [ ]:
import torch

from bci_eeg_to_text.backbones import build_ctc_model

print("torch", torch.__version__)
print("cuda  ", torch.cuda.is_available(), "(expected False on a hosted image)")

In [ ]:
# A tiny vocabulary standing in for the real one.
from bci_eeg_to_text.tokenizer import CharTokenizer

tok = CharTokenizer.build(["我六岁那年", "看见过一幅精彩的插图"])
print("classes:", len(tok))
text = "我六岁那年"
ids = tok.encode(text)
print("encode:", ids)
print("decode:", tok.decode(ids), "| roundtrip ok:", tok.decode(ids) == text)

In [ ]:
# Build a CTC model and run one forward pass on synthetic EEG.
import torch

model = build_ctc_model("scratch", n_channels=8, n_classes=len(tok), d_model=64)
eeg = torch.randn(2, 8, 4096)
logits = model(eeg)
frames = model.input_lengths_to_output(torch.tensor([4096]))
summary = model.parameter_summary()

print("eeg      ", tuple(eeg.shape))
print("logits   ", tuple(logits.shape))
print("frames   ", int(frames[0]), "(must be >= target length for CTC)")
print(f"params    {summary['trainable']:,.0f}")

## Decoding

Greedy decoding collapses repeats unless a blank separates them.


In [ ]:
from bci_eeg_to_text.decode import beam_search_decode, greedy_decode

a = tok.stoi["我"]
paths = torch.full((1, 5, len(tok)), -10.0)
# 我, blank, 我 -> two characters, because blank separates them.
for t, c in enumerate([a, tok.blank, a, tok.blank, tok.blank]):
    paths[0, t, c] = 10.0

print("greedy:", greedy_decode(paths, tok))
print("beam  :", beam_search_decode(paths[0], tok, beam_width=8))

## Leakage modes

A split can leak text even when the recordings are disjoint. These are the
modes the CLI exposes; see the README for the full matrix.


In [ ]:
from bci_eeg_to_text.leakage import (
    LEAKAGE_MODES,
    SUBJECT_MODES,
    describe_subject_mode,
    describe_text_mode,
)

print("text modes:")
for m in LEAKAGE_MODES:
    print(f"  {m:10} {describe_text_mode(m)}")
print("subject modes:")
for m in SUBJECT_MODES:
    print(f"  {m:14} {describe_subject_mode(m)}")

## Using real data

Fetch a corpus with DataLad, then point the CLI at it:

```bash
export EEG_TEXT_BCI_DATA_ROOT=/path/to/bci-eeg
uv run bci-eeg-to-text split --dataset chineseeeg --link
uv run bci-eeg-to-text train --split splits --out runs/exp1
```
